# 04 · Support Vector Regression

Linear, degree-3 polynomial and RBF kernels. Hyper-parameters are tuned by `GridSearchCV` **on the training split only** (never on the test set), then evaluated once on the held-out 30%.

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import punching_shear as ps
REPO = Path(ps.__file__).resolve().parent.parent
RESULTS = REPO / 'results'
ds = ps.load_dataset()
print(f'{len(ds)} tests, {ds.groups.nunique()} researchers, features={ds.feature_names}')

336 tests, 55 researchers, features=['d', 'col_area', 'rho_l', 'fcm_cyl', 'u0_perim']


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone
models = ps.build_models()
svr = ['SVR (linear)','SVR (poly-3)','SVR (RBF)']
Xtr,Xte,ytr,yte = train_test_split(ds.X, ds.y_stress, test_size=.3, random_state=19)
out=[]
for name in svr:
    gs=clone(models[name]).fit(Xtr,ytr)
    best=gs.best_estimator_.named_steps['m']
    nsv=int(best.support_.shape[0])
    m=ps.regression_metrics(yte, gs.predict(Xte))
    out.append({'model':name,'rmse':m['rmse'],'r2':m['r2'],'n_SV':nsv,'best':gs.best_params_})
display(pd.DataFrame(out))

,model,rmse,r2,n_SV,best
0,SVR (linear),0.306956,0.582825,149,"{'m__C': 5, 'm__epsilon': 0.1}"
1,SVR (poly-3),0.406242,0.269304,207,"{'m__C': 1, 'm__epsilon': 0.05, 'm__gamma': 0.1}"
2,SVR (RBF),0.276070,0.662555,141,"{'m__C': 10, 'm__epsilon': 0.1, 'm__gamma': 0.05}"


### Cross-validated SVR numbers (from `results/`)

In [3]:
rand = pd.read_csv(RESULTS/'metrics_random_kfold.csv')
display(rand[rand.model.isin(svr)][['model','rmse_mean','rmse_ci95','r2_mean']].round(4))

,model,rmse_mean,rmse_ci95,r2_mean
1,SVR (RBF),0.2521,0.0144,0.7521
7,SVR (linear),0.2883,0.0139,0.6746
10,SVR (poly-3),0.6554,0.1364,-0.9011


**Takeaway.** The **RBF** kernel is the strongest SVR (CV R² ≈ 0.75) and the only one competitive with Random Forest. The **polynomial** kernel overfits badly (negative R²). But note: this is on *random* splits — see notebook 06 for what happens under researcher-held-out validation.